In [1]:
!pip install -q google-genai google-cloud-aiplatform[evaluation] pytest pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 16.0 MB/s eta 0:00:00


In [2]:
PROJECT_ID = "qwiklabs-gcp-02-5d502ebd5e20"
LOCATION = "us-central1"
MODEL_NAME = "gemini-2.5-flash"

In [3]:
from google import genai
from google.genai import types

client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION
)

In [4]:
CATEGORIES = [
    "complaint",
    "question",
    "feedback",
    "general information"
]

In [5]:
def classify_message(message: str) -> str:
    prompt = f"""
You are a message classification system.

Classify the message into exactly ONE category:
- complaint
- question
- feedback
- general information

Definitions:
- complaint: expresses dissatisfaction or a problem
- question: asks for information
- feedback: opinions or comments without dissatisfaction
- general information: anything not fitting above

Return ONLY the category name.

Message:
{message}

Category:
"""
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt
    )

    category = response.text.strip().lower()
    return category if category in CATEGORIES else "general information"

In [47]:
test_messages = [
    "Your service is terrible",
    "What time does city hall open?",
    "I like the new website design",
    "City hall is downtown"
]

for msg in test_messages:
    print(msg, "=>", classify_message(msg))

Your service is terrible => complaint
What time does city hall open? => question
I like the new website design => feedback
City hall is downtown => general information


In [7]:
def generate_social_post(event_type: str, details: str) -> str:
    prompt = f"""
You are a social media manager for the government. You handle communications.

Write clear and concise social media posts for the public.

Event type: {event_type}
Details: {details}

Requirements:
- Under 280 characters
- Professional and calm tone
- Plain language
- No speculation or panic
- Return ONLY the post text
"""
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt
    )

    return response.text.strip()

In [12]:
print(generate_social_post(
    "Holiday Notification",
    "Reminder about upcoming holiday on May 25 for Memorial Day."
))

Reminder: Memorial Day is Monday, May 25. Most government offices will be closed. Please plan accordingly.


In [45]:
def test_classify_complaint():
    assert classify_message("Your service is unacceptable") == "complaint"

def test_classify_question():
    assert classify_message("How do I apply for a permit?") == "question"

def test_classify_feedback():
    assert classify_message("The website looks great") == "feedback"

def test_classify_general_info():
    assert classify_message("City hall is downtown") == "general information"

def test_social_post_length():
    post = generate_social_post(
        "Holiday Notification",
        "City offices will be closed Monday for Memorial Day."
    )
    assert len(post) <= 280

def test_social_post_not_empty():
    post = generate_social_post(
        "Weather Emergency",
        "Heavy snow expected tonight. Avoid unnecessary travel."
    )
    assert len(post.strip()) > 0


# Execute tests
test_classify_complaint()
test_classify_question()
test_classify_feedback()
test_classify_general_info()
test_social_post_length()
test_social_post_not_empty()

print("✅ All tests passed")


✅ All tests passed


In [18]:
import pandas as pd
import vertexai
from vertexai.evaluation import EvalTask, PointwiseMetric, PointwiseMetricPromptTemplate

In [27]:
vertexai.init(project=PROJECT_ID, location=LOCATION)


In [81]:
def social_post_prompt_v1(event_type: str, details: str) -> str:
    return f"""
Write a short social media post.

"""

In [49]:
def social_post_prompt_v2(event_type: str, details: str) -> str:
    return f"""
You are a social media manager for the government. You handle communications.

Write clear and concise social media posts for the public.

Event type: {event_type}
Details: {details}

Requirements:
- Under 280 characters
- Professional and calm tone
- Plain language
- No speculation or panic
- Return ONLY the post text
"""

In [85]:
eval_inputs = [
    {
        "event_type": "Weather Emergency",
        "details": "Freezing rain expected overnight. Roads may be hazardous."
    },
    {
        "event_type": "School Closing",
        "details": "All public schools are closed tomorrow due to severe snowfall."
    },
    {
        "event_type": "Holiday Closure",
        "details": "City offices will be closed Monday for the statutory holiday."
    }
]

results = []

for row in eval_inputs:
    response_v1 = client.models.generate_content(
        model=MODEL_NAME,
        contents=social_post_prompt_v1(row["event_type"], row["details"])
    ).text.strip()

    response_v2 = client.models.generate_content(
        model=MODEL_NAME,
        contents=social_post_prompt_v2(row["event_type"], row["details"])
    ).text.strip()

    results.append({
        "prompt_v1_response": response_v1,
        "prompt_v2_response": response_v2
    })

In [86]:
eval_df = pd.DataFrame(results)
eval_df

,prompt_v1_response,prompt_v2_response
0,"Here are a few options, choose the one that be...",Freezing rain expected overnight. Roads may be...
1,"Here are a few options, choose the one that be...",All public schools will be closed tomorrow due...
2,Here are a few options for a short social medi...,City offices will be closed Monday for the sta...


In [87]:
social_post_quality = PointwiseMetric(
    metric="social_post_quality",
    metric_prompt_template=PointwiseMetricPromptTemplate(
        criteria={
            "clarity": "The post is easy for the public to understand.",
            "tone": "The post must sound explicitly like an official government announcement and not casual, conversational, or generic.",
            "actionability": "The post provides useful action steps when relevant.",
            "brevity": "The post is short and suitable for social media."
        },
        rating_rubric={
            "5": "Excellent government announcement.",
            "4": "Good announcement with minor issues.",
            "3": "Adequate but could be clearer.",
            "2": "Weak announcement.",
            "1": "Poor announcement."
        }
    )
)

INFO:vertexai.evaluation.metrics.metric_prompt_template:The `input_variables` parameter is empty. Only the `response` column is used for computing this model-based metric.


In [88]:
eval_task_v1 = EvalTask(
    dataset=eval_df.rename(columns={"prompt_v1_response": "response"}),
    metrics=[social_post_quality],
    experiment="task3-social-post-eval-v1"
)

eval_result_v1 = eval_task_v1.evaluate()
eval_result_v1.summary_metrics

INFO:vertexai.evaluation._evaluation:Computing metrics with a total of 3 Vertex Gen AI Evaluation Service API requests.
100%|██████████| 3/3 [00:08<00:00,  2.96s/it]
INFO:vertexai.evaluation._evaluation:All 3 metric requests are successfully computed.
INFO:vertexai.evaluation._evaluation:Evaluation Took:8.87764895899818 seconds


{'row_count': 3,
 'social_post_quality/mean': np.float64(1.0),
 'social_post_quality/std': 0.0}

In [89]:
eval_result_v1.metrics_table

,response,prompt_v2_response,social_post_quality/explanation,social_post_quality/score
0,"Here are a few options, choose the one that be...",Freezing rain expected overnight. Roads may be...,The AI response completely fails on the 'tone'...,1.0
1,"Here are a few options, choose the one that be...",All public schools will be closed tomorrow due...,The response completely fails on the 'tone' cr...,1.0
2,Here are a few options for a short social medi...,City offices will be closed Monday for the sta...,The AI-generated response completely fails to ...,1.0


In [90]:
eval_task_v2 = EvalTask(
    dataset=eval_df.rename(columns={"prompt_v2_response": "response"}),
    metrics=[social_post_quality],
    experiment="task3-social-post-eval-v2"
)

eval_result_v2 = eval_task_v2.evaluate()
eval_result_v2.summary_metrics


INFO:vertexai.evaluation._evaluation:Computing metrics with a total of 3 Vertex Gen AI Evaluation Service API requests.
100%|██████████| 3/3 [00:08<00:00,  2.88s/it]
INFO:vertexai.evaluation._evaluation:All 3 metric requests are successfully computed.
INFO:vertexai.evaluation._evaluation:Evaluation Took:8.640717800000857 seconds


{'row_count': 3,
 'social_post_quality/mean': np.float64(5.0),
 'social_post_quality/std': 0.0}

In [91]:
eval_result_v2.metrics_table

,prompt_v1_response,response,social_post_quality/explanation,social_post_quality/score
0,"Here are a few options, choose the one that be...",Freezing rain expected overnight. Roads may be...,"The announcement is excellent; it is brief, cl...",5.0
1,"Here are a few options, choose the one that be...",All public schools will be closed tomorrow due...,The announcement is excellent as it is perfect...,5.0
2,Here are a few options for a short social medi...,City offices will be closed Monday for the sta...,The announcement is excellent; it is perfectly...,5.0


### Prompt 2 which contains clearer and better constraints turned out to be the preferred prompt based on the evaluation metrics.